1. Load Libraries

In [2]:
%pip install -q langchain langgraph langchain-groq pydantic python-dotenv duckduckgo-search ddgs
import os
import json
import requests
from typing import Annotated, TypedDict , Literal
from dotenv import load_dotenv

# LangChain Core & Groq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

# LangGraph Core Components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in your .env file")


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


2. Define State

In [3]:
class MultiAgentState(TypedDict):
    user_query: str
    math_result: str | None
    trivia_result: str | None
    chat_result: str | None
    messages: list[dict]  # [{"role": "user"/"assistant", "content": "..."}]

3. Nodes

In [4]:
def math_node(state: MultiAgentState):
    question = state["user_query"]
    try:
        answer = eval(question)
        state["math_result"] = str(answer)
        state["messages"].append({"role": "assistant", "content": f"Answer: {answer}"})
    except Exception:
        state["math_result"] = "Error"
        state["messages"].append({"role": "assistant", "content": "Sorry, I can only do simple math."})
    return state

def trivia_node(state: MultiAgentState):
    question = state["user_query"].lower()
    if "capital of france" in question:
        answer = "Paris"
    elif "largest planet" in question:
        answer = "Jupiter"
    else:
        answer = "I don't know that trivia yet."
    state["trivia_result"] = answer
    state["messages"].append({"role": "assistant", "content": answer})
    return state

def chat_node(state: MultiAgentState):
    user_text = state["user_query"]
    reply = f"I’m just chatting with you: '{user_text}'"
    state["chat_result"] = reply
    state["messages"].append({"role": "assistant", "content": reply})
    return state

4. Router Node

In [5]:
def router(state: MultiAgentState):
    text = state["user_query"].lower()
    if any(char.isdigit() for char in text):
        return "math"
    elif "capital" in text or "planet" in text:
        return "trivia"
    else:
        return "chat"

5. Build Graph

In [6]:
graph = StateGraph(MultiAgentState)

graph.add_node("math", math_node)
graph.add_node("trivia", trivia_node)
graph.add_node("chat", chat_node)

# Entry point is "chat", but we immediately branch using router
graph.set_entry_point("chat")

graph.add_conditional_edges(
    "chat",
    router,
    {
        "math": "math",
        "trivia": "trivia",
        "chat": END
    }
)

graph.add_edge("math", END)
graph.add_edge("trivia", END)
graph.add_edge("chat", END)

6. Compiling Runtime

In [7]:
runtime = graph.compile(checkpointer=InMemorySaver())

7. Examples

In [8]:
config = {"configurable": {"thread_id": "demo-thread"}}

# Example 1: Math
result = runtime.invoke(
    {"user_query": "12 * 7", "messages": [{"role": "user", "content": "12 * 7"}]},
    config=config
)
print(result["messages"][-1]["content"])   # → Answer: 84

# Example 2: Trivia
result = runtime.invoke(
    {"user_query": "What is the largest planet?", "messages": [{"role": "user", "content": "What is the largest planet?"}]},
    config=config
)
print(result["messages"][-1]["content"])   # → Jupiter

# Example 3: Chat
result = runtime.invoke(
    {"user_query": "Hello, how are you?", "messages": [{"role": "user", "content": "Hello, how are you?"}]},
    config=config
)
print(result["messages"][-1]["content"])  # → I’m just chatting with you: 'Hello, how are you?'

Answer: 84
Jupiter
I’m just chatting with you: 'Hello, how are you?'
